# clip-grad-norm-pre-step — faded example 1: Insert the clip call in the right place in a training step

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `clip-grad-norm-pre-step`. Running the beacon reports progress on the `Optimizer: clip_grad_norm pre-step` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Optimizer: clip_grad_norm pre-step` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`clip-grad-norm-pre-step`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "clip-grad-norm-pre-step"
DD_SUBTOPIC = "Optimizer: clip_grad_norm pre-step"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In a stable training loop, gradient clipping happens after `loss.backward()` and before `optimizer.step()`. `torch.nn.utils.clip_grad_norm_(params, max_norm)` rescales the global gradient norm in place and returns the pre-clip norm. Misplacing it (before backward, or after step) makes it useless.

## Faded exercise 1

### Faded - place the clip call

Complete `clipped_step` so that the model's gradients are clipped to a global L2 norm of `max_norm` between the backward pass and the optimizer step. The function must return the pre-clip norm as a Python float. Fill in the one missing line.

**Fill in:** Clip the global gradient norm of the model's parameters to max_norm and capture the returned pre-clip norm tensor.

In [ ]:
import torch.nn as nn
import torch.nn.utils as nn_utils

t.manual_seed(0)

def clipped_step(model, optimizer, x, y, max_norm):
    optimizer.zero_grad()
    pred = model(x)
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    pre_norm = nn_utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
    optimizer.step()
    optimizer.zero_grad()
    return pre_norm.item()

model = nn.Linear(4, 1)
opt = t.optim.SGD(model.parameters(), lr=0.1)
x = t.randn(8, 4)
y = t.randn(8, 1)
result = clipped_step(model, opt, x, y, max_norm=1.0)
print("pre-clip norm:", round(result, 4))


def _test():
    import torch.nn as nn
    import torch.nn.utils as nn_utils
    t.manual_seed(0)
    model = nn.Linear(4, 1)
    opt = t.optim.SGD(model.parameters(), lr=0.1)
    x = t.randn(8, 4)
    y = t.randn(8, 1)
    # independent ground truth: replicate the SAME RNG consumption order as above
    # (seed -> build Linear -> draw x -> draw y), then measure norm without clipping
    t.manual_seed(0)
    model_ref = nn.Linear(4, 1)
    opt_ref = t.optim.SGD(model_ref.parameters(), lr=0.1)
    x_ref = t.randn(8, 4)
    y_ref = t.randn(8, 1)
    opt_ref.zero_grad()
    pred_ref = model_ref(x_ref)
    loss_ref = ((pred_ref - y_ref) ** 2).mean()
    loss_ref.backward()
    expected = nn_utils.clip_grad_norm_(model_ref.parameters(), max_norm=1.0).item()
    got = clipped_step(model, opt, x, y, max_norm=1.0)
    assert isinstance(got, float), "must return a Python float"
    assert abs(got - expected) < 1e-5, f"expected pre-clip norm {expected}, got {got}"
    # after a clipped step, grads should be cleared
    for p in model.parameters():
        assert p.grad is None or t.all(p.grad == 0), "grads should be zeroed after step"


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch.nn as nn
import torch.nn.utils as nn_utils

t.manual_seed(0)

def clipped_step(model, optimizer, x, y, max_norm):
    optimizer.zero_grad()
    pred = model(x)
    loss = ((pred - y) ** 2).mean()
    loss.backward()
    pre_norm = nn_utils.clip_grad_norm_(model.parameters(), max_norm=max_norm)
    optimizer.step()
    optimizer.zero_grad()
    return pre_norm.item()

model = nn.Linear(4, 1)
opt = t.optim.SGD(model.parameters(), lr=0.1)
x = t.randn(8, 4)
y = t.randn(8, 1)
result = clipped_step(model, opt, x, y, max_norm=1.0)
print("pre-clip norm:", round(result, 4))
```
</details>